<a href="https://colab.research.google.com/github/csabiu/Cosmology_Course/blob/main/practical/dark_energy_supernovae.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/csabiu/Cosmology_Course/blob/main/practical/dark_energy_supernovae.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dark Energy Constraints from Type Ia Supernovae
### A Computational Tutorial for Lecture 3 — Distances in Cosmology and Dark Energy

---

## Introduction

In Lectures 1 and 2, we measured the Hubble constant and derived the Friedmann equations. Now we ask: *how do we know the expansion is accelerating, and what is driving it?*

The answer came in 1998 when two teams (the Supernova Cosmology Project and the High-z Supernova Search Team) discovered that distant Type Ia supernovae are **fainter than expected** in a decelerating universe. This implied the expansion is accelerating, driven by a mysterious **dark energy** component.

In this notebook you will:
1. **Compute cosmological distances** (comoving, luminosity, angular diameter) from the Friedmann equation and verify them against `astropy`
2. **Download real Type Ia supernova data** from the Pantheon compilation (1048 SNe Ia) and build the Hubble diagram
3. **Fit cosmological models** to the data using $\chi^2$ with the full systematic covariance matrix
4. **Run MCMC** with `emcee` to obtain Bayesian posterior constraints on $\Omega_m$, $w$, $w_0$, and $w_a$
5. **Compare models** ($\Lambda$CDM vs $w$CDM vs CPL) using the Akaike and Bayesian Information Criteria

### Prerequisites
Lectures 1–3, basic Python (numpy, matplotlib, scipy). MCMC sampling uses [emcee](https://emcee.readthedocs.io/). Corner plots use [corner](https://corner.readthedocs.io/).

### Key Equations

**Distance modulus:**
$$\mu(z) = 5\log_{10}\!\left(\frac{d_L(z)}{\mathrm{Mpc}}\right) + 25$$

**Generalised chi-squared with covariance:**
$$\chi^2(\boldsymbol{\theta}) = \Delta\boldsymbol{\mu}^{\!\top}\, \mathbf{C}^{-1}\, \Delta\boldsymbol{\mu}, \qquad \Delta\mu_i = m_{B,i} - \mu_{\mathrm{th}}(z_i;\,\boldsymbol{\theta})$$

**Bayes’ theorem:**
$$P(\boldsymbol{\theta}\,|\,\mathbf{d},\,M) = \frac{\mathcal{L}(\mathbf{d}\,|\,\boldsymbol{\theta},\,M)\;\pi(\boldsymbol{\theta}\,|\,M)}{\mathcal{Z}(\mathbf{d}\,|\,M)}$$

---

## Part 0 — Setup

Install any packages not available in the default environment (needed on Google Colab).

In [ ]:
# Install packages needed on Google Colab
# (skip if running locally with these already installed)
!pip install -q emcee corner astropy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.integrate import quad
from scipy.optimize import minimize, differential_evolution
from astropy.cosmology import FlatLambdaCDM, Flatw0waCDM
import emcee
import corner
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

# Nice plot defaults (matching course style)
plt.rcParams.update({
    'font.size': 13,
    'axes.labelsize': 14,
    'axes.titlesize': 15,
    'legend.fontsize': 11,
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# Physical constants
c_km_s = 2.998e5   # speed of light in km/s
H0_fid = 70.0      # fiducial Hubble constant (km/s/Mpc) — fixed throughout

print('Libraries loaded successfully.')

---
## Part 1 — Cosmological Distances

In Lecture 3 we derived the three fundamental cosmological distance measures from the FLRW metric. All of them depend on the **comoving distance**:

$$\chi(z) = \frac{c}{H_0}\int_0^z \frac{\mathrm{d}z'}{E(z')}$$

where $E(z) \equiv H(z)/H_0$ is the **dimensionless Hubble parameter**. For a flat $\Lambda$CDM universe:

$$E(z) = \sqrt{\Omega_m(1+z)^3 + \Omega_\Lambda}$$

From $\chi(z)$ we compute:
- **Luminosity distance:** $d_L(z) = (1+z)\,\chi(z)$ — relates observed flux to intrinsic luminosity
- **Angular diameter distance:** $d_A(z) = \chi(z)/(1+z)$ — relates observed angle to physical size
- **Distance modulus:** $\mu(z) = 5\log_{10}(d_L/\mathrm{Mpc}) + 25$ — what we measure for supernovae

In [ ]:
# ============================================================
# Distance functions for flat LCDM
# ============================================================

def E_z_LCDM(z, Om):
    """Dimensionless Hubble parameter E(z) = H(z)/H0 for flat LCDM."""
    return np.sqrt(Om * (1 + z)**3 + (1 - Om))

def comoving_distance(z, Om, H0=70.0):
    """Comoving distance chi(z) in Mpc for flat LCDM."""
    integrand = lambda zp: 1.0 / E_z_LCDM(zp, Om)
    result, _ = quad(integrand, 0, z)
    return c_km_s / H0 * result

def luminosity_distance(z, Om, H0=70.0):
    """Luminosity distance dL(z) in Mpc for flat LCDM."""
    return (1 + z) * comoving_distance(z, Om, H0)

def angular_diameter_distance(z, Om, H0=70.0):
    """Angular diameter distance dA(z) in Mpc for flat LCDM."""
    return comoving_distance(z, Om, H0) / (1 + z)

def distance_modulus_calc(z, Om, H0=70.0):
    """Distance modulus mu(z) in magnitudes for flat LCDM."""
    dL = luminosity_distance(z, Om, H0)
    return 5.0 * np.log10(dL) + 25.0

# Quick test
print(f"At z=1, Om=0.3: dL = {luminosity_distance(1.0, 0.3):.1f} Mpc")
print(f"                 dA = {angular_diameter_distance(1.0, 0.3):.1f} Mpc")
print(f"                 mu = {distance_modulus_calc(1.0, 0.3):.2f} mag")

In [ ]:
# ============================================================
# Plot all three distances for different matter densities
# ============================================================
z_arr = np.linspace(0.01, 5.0, 200)
Om_values = [0.1, 0.3, 0.5, 1.0]
colors = ['darkorange', 'steelblue', 'seagreen', 'crimson']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for Om, col in zip(Om_values, colors):
    dL_arr = np.array([luminosity_distance(z, Om) for z in z_arr])
    dA_arr = np.array([angular_diameter_distance(z, Om) for z in z_arr])
    mu_arr = np.array([distance_modulus_calc(z, Om) for z in z_arr])

    label = rf'$\Omega_m = {Om}$'
    axes[0].plot(z_arr, dL_arr, color=col, lw=2, label=label)
    axes[1].plot(z_arr, dA_arr, color=col, lw=2, label=label)
    axes[2].plot(z_arr, mu_arr, color=col, lw=2, label=label)

axes[0].set_ylabel('$d_L$ (Mpc)')
axes[0].set_title('Luminosity Distance')
axes[1].set_ylabel('$d_A$ (Mpc)')
axes[1].set_title('Angular Diameter Distance')
axes[2].set_ylabel(r'$\mu$ (mag)')
axes[2].set_title('Distance Modulus')

for ax in axes:
    ax.set_xlabel('Redshift $z$')
    ax.legend(fontsize=9)

plt.suptitle('Cosmological Distances in Flat $\\Lambda$CDM', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('cosmological_distances_tutorial.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================
# Verify against astropy's built-in functions
# ============================================================
cosmo_ref = FlatLambdaCDM(H0=70, Om0=0.3)

z_test = [0.1, 0.5, 1.0, 2.0]
print(f"{'z':>5} {'dL (ours)':>12} {'dL (astropy)':>14} {'diff (Mpc)':>12}")
print('-' * 48)
for z in z_test:
    dL_ours = luminosity_distance(z, 0.3, 70.0)
    dL_astropy = cosmo_ref.luminosity_distance(z).value
    print(f"{z:>5.1f} {dL_ours:>12.2f} {dL_astropy:>14.2f} {abs(dL_ours - dL_astropy):>12.4f}")

print("\nAgreement to < 0.01 Mpc — our implementation is correct!")

### Exercise 1: Etherington Reciprocity Relation

The **Etherington reciprocity relation** (1933) states that in *any* metric theory of gravity that conserves photon number:

$$d_L(z) = (1+z)^2\, d_A(z)$$

This is a powerful model-independent relation. Violations would signal exotic physics (e.g. photon–axion oscillations or variable fundamental constants).

**Task:** Verify numerically that $d_L = (1+z)^2 \, d_A$ for a range of redshifts.

In [ ]:
# YOUR CODE HERE: Verify the Etherington reciprocity relation
#
# Task: Show that dL = (1+z)^2 * dA for z = 0.01, 0.1, 0.5, 1.0, 2.0, 5.0
#
# Hints:
#   - Use the luminosity_distance() and angular_diameter_distance() functions from above
#   - Compute the ratio dL / ((1+z)^2 * dA) and verify it equals 1.0
#   - Print a formatted table and make a plot of dL vs (1+z)^2 * dA


---
## Part 2 — The Pantheon Type Ia Supernova Dataset

Type Ia supernovae are thermonuclear explosions of carbon–oxygen white dwarfs near the Chandrasekhar mass ($\sim 1.4\,M_\odot$). Their intrinsic scatter in peak luminosity is reduced to $\sigma_M \approx 0.1$–0.15 mag by the **Phillips relation** (1993): brighter supernovae decline more slowly.

The **Pantheon** sample (Scolnic et al. 2018) compiles **1048 spectroscopically confirmed SNe Ia** from multiple surveys, spanning $0.01 < z < 2.3$. For each supernova we have:
- $z_{\mathrm{cmb}}$: redshift in the CMB rest frame
- $m_B$: apparent B-band magnitude (corrected for stretch and colour)
- $\sigma_{m_B}$: statistical uncertainty on $m_B$

The theoretical prediction is:
$$m_{B,\mathrm{th}}(z) = \mu(z;\,\boldsymbol{\theta}) + M_B$$
where $M_B$ is the absolute magnitude (a nuisance parameter degenerate with $H_0$).

In [ ]:
# ============================================================
# Download the Pantheon dataset
# ============================================================
url_data = "https://raw.githubusercontent.com/dscolnic/Pantheon/master/lcparam_full_long.txt"
pan = pd.read_csv(url_data, delim_whitespace=True)

# Extract the columns we need
zcmb = pan['zcmb'].values     # CMB-frame redshift
mb   = pan['mb'].values       # corrected apparent magnitude
dmb  = pan['dmb'].values      # statistical uncertainty

N_sne = len(zcmb)
print(f"Pantheon dataset: {N_sne} Type Ia supernovae")
print(f"Redshift range:   {zcmb.min():.4f} – {zcmb.max():.3f}")
print(f"Magnitude range:  {mb.min():.2f} – {mb.max():.2f}")
print(f"Typical error:    {np.median(dmb):.3f} mag")

In [ ]:
# ============================================================
# Hubble diagram: apparent magnitude vs redshift
# ============================================================
MB_fid = -19.35   # fiducial absolute magnitude

fig, ax = plt.subplots(figsize=(10, 7))

# Data points (colour-coded by redshift)
sc = ax.scatter(zcmb, mb, c=zcmb, cmap='viridis', s=8, alpha=0.6,
                edgecolors='none', zorder=3, label='Pantheon data')
plt.colorbar(sc, ax=ax, label='Redshift $z$', shrink=0.8)

# Theoretical curves
z_th = np.linspace(0.01, 2.5, 300)

# Einstein-de Sitter (matter-only, decelerating)
cosmo_EdS = FlatLambdaCDM(H0=H0_fid, Om0=1.0)
ax.plot(z_th, cosmo_EdS.distmod(z_th).value + MB_fid, 'crimson', lw=2, ls='--',
        label=r'Einstein–de Sitter ($\Omega_m=1$)')

# Concordance LCDM
cosmo_LCDM = FlatLambdaCDM(H0=H0_fid, Om0=0.3)
ax.plot(z_th, cosmo_LCDM.distmod(z_th).value + MB_fid, 'navy', lw=2.5,
        label=r'$\Lambda$CDM ($\Omega_m=0.3,\;\Omega_\Lambda=0.7$)')

# Open universe (no Lambda)
cosmo_open = FlatLambdaCDM(H0=H0_fid, Om0=0.3)  # For comparison use low-density flat
# Actually use a matter-only open model via manual calculation
from astropy.cosmology import LambdaCDM
cosmo_open = LambdaCDM(H0=H0_fid, Om0=0.3, Ode0=0.0)
ax.plot(z_th, cosmo_open.distmod(z_th).value + MB_fid, 'darkorange', lw=2, ls=':',
        label=r'Open ($\Omega_m=0.3,\;\Omega_\Lambda=0$)')

ax.set_xscale('log')
ax.set_xlabel('Redshift $z$')
ax.set_ylabel('Apparent magnitude $m_B$')
ax.set_title('Hubble Diagram: 1048 Pantheon Type Ia Supernovae')
ax.legend(loc='upper left', fontsize=10)
ax.set_xlim(0.008, 3.0)
ax.invert_yaxis()  # Brighter = smaller magnitude

plt.tight_layout()
plt.savefig('pantheon_hubble_diagram.png', bbox_inches='tight', dpi=150)
plt.show()

### Interpreting the Hubble Diagram

The key observation: at $z \gtrsim 0.5$, the data points lie **above** the Einstein–de Sitter and open-universe curves. This means distant supernovae are **fainter** (larger $m_B$, hence larger $d_L$) than predicted by a decelerating universe. The concordance $\Lambda$CDM model ($\Omega_m = 0.3$, $\Omega_\Lambda = 0.7$) fits the data well.

This was the discovery of **cosmic acceleration** (Perlmutter et al. 1999; Riess et al. 1998), awarded the 2011 Nobel Prize in Physics.

---
## Part 3 — Chi-Squared with the Covariance Matrix

To extract cosmological parameters, we minimise the **generalised chi-squared**:

$$\chi^2(\boldsymbol{\theta}) = \Delta\boldsymbol{\mu}^{\top} \, \mathbf{C}^{-1} \, \Delta\boldsymbol{\mu}$$

where $\Delta\mu_i = m_{B,i} - \mu_{\mathrm{th}}(z_i;\,\boldsymbol{\theta}) - M_B$ and $\mathbf{C}$ is the **total covariance matrix**.

Why do we need a covariance matrix instead of just individual error bars? Because the 1048 SNe Ia share **systematic uncertainties**: photometric calibration, light-curve model training, dust corrections, survey-specific biases, etc. These create **correlations** between the distance modulus errors of different supernovae. Ignoring these correlations leads to incorrect parameter estimates and underestimated uncertainties.

The total covariance is:
$$\mathbf{C}_{\mathrm{tot}} = \mathbf{C}_{\mathrm{sys}} + \mathrm{diag}(\sigma_{m_B}^2)$$

In [ ]:
# ============================================================
# Download the Pantheon systematic covariance matrix
# ============================================================
print("Downloading systematic covariance matrix (~10 MB)...")
url_cov = "https://raw.githubusercontent.com/dscolnic/Pantheon/master/sys_full_long.txt"
raw_cov = np.loadtxt(url_cov)

# The file contains a header value followed by the flattened 1048x1048 matrix
Csys = raw_cov[1:].reshape((N_sne, N_sne))

# Total covariance = systematic + statistical (diagonal)
Ctot = Csys + np.diag(dmb**2)

# Precompute the precision (inverse covariance) matrix
icov = np.linalg.inv(Ctot)

print(f"Covariance matrix shape: {Ctot.shape}")
print(f"Condition number:        {np.linalg.cond(Ctot):.2e}")
print("Done.")

In [ ]:
# ============================================================
# Visualise the covariance and correlation matrices
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: log|C| heatmap
im1 = ax1.imshow(np.log10(np.abs(Ctot) + 1e-10), cmap='inferno', aspect='auto')
ax1.set_title(r'$\log_{10}|\mathbf{C}_{\mathrm{tot}}|$')
ax1.set_xlabel('SN index')
ax1.set_ylabel('SN index')
plt.colorbar(im1, ax=ax1, shrink=0.8)

# Right: correlation matrix R_ij = C_ij / sqrt(C_ii * C_jj)
diag = np.sqrt(np.diag(Ctot))
Rcorr = Ctot / np.outer(diag, diag)

im2 = ax2.imshow(Rcorr, cmap='RdBu_r', vmin=-0.3, vmax=0.3, aspect='auto')
ax2.set_title('Correlation Matrix $R_{ij}$')
ax2.set_xlabel('SN index')
ax2.set_ylabel('SN index')
plt.colorbar(im2, ax=ax2, shrink=0.8)

plt.suptitle('Pantheon Covariance Structure', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('covariance_matrix_visualisation.png', bbox_inches='tight', dpi=150)
plt.show()

print(f"Off-diagonal correlations range from {Rcorr[np.triu_indices(N_sne, k=1)].min():.3f} "
      f"to {Rcorr[np.triu_indices(N_sne, k=1)].max():.3f}")
print("The block structure reflects SNe from different surveys sharing calibration systematics.")

In [ ]:
# ============================================================
# Chi-squared function for flat LCDM
# Parameters: [Om, MB]
# H0 is fixed at 70 km/s/Mpc (degenerate with MB for SNe)
# ============================================================

def chi2_LCDM(params, zcmb, mb, icov):
    """Chi-squared for flat LCDM: params = [Om, MB]."""
    Om, MB = params
    if Om <= 0 or Om >= 1:
        return 1e10
    cosmo = FlatLambdaCDM(H0=H0_fid, Om0=Om)
    mu_th = cosmo.distmod(zcmb).value
    dy = mb - mu_th - MB
    return float(dy @ icov @ dy)

# Quick sanity check
chi2_test = chi2_LCDM([0.3, -19.35], zcmb, mb, icov)
print(f"chi2(Om=0.3, MB=-19.35) = {chi2_test:.1f}")
print(f"chi2_red = {chi2_test / (N_sne - 2):.4f}")

In [ ]:
# ============================================================
# Find best-fit parameters using global optimisation
# ============================================================
print("Finding best-fit flat LCDM parameters...")
bounds_LCDM = [(0.01, 0.99), (-25.0, -15.0)]
res_LCDM = differential_evolution(
    chi2_LCDM, bounds_LCDM, args=(zcmb, mb, icov),
    seed=42, tol=1e-8, maxiter=1000
)

Om_best, MB_best = res_LCDM.x
chi2_min_LCDM = res_LCDM.fun

print(f"\nBest-fit flat LCDM:")
print(f"  Om = {Om_best:.4f}")
print(f"  MB = {MB_best:.4f}")
print(f"  chi2_min = {chi2_min_LCDM:.2f}")
print(f"  chi2_red = {chi2_min_LCDM / (N_sne - 2):.4f}")
print(f"  (N_data = {N_sne}, N_params = 2)")

In [ ]:
# ============================================================
# Chi-squared profile: scan Om, minimise MB at each point
# ============================================================
Om_grid = np.linspace(0.05, 0.80, 60)
chi2_profile = np.empty_like(Om_grid)

for i, Om in enumerate(Om_grid):
    # For each Om, find the MB that minimises chi2
    res_mb = minimize(lambda MB: chi2_LCDM([Om, MB[0]], zcmb, mb, icov),
                      x0=[MB_best], method='Nelder-Mead')
    chi2_profile[i] = res_mb.fun

delta_chi2 = chi2_profile - chi2_min_LCDM

# --- Plot ---
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(Om_grid, delta_chi2, 'navy', lw=2.5)

# Confidence levels
ax.axhline(1.0, color='steelblue', ls='--', lw=1.5, alpha=0.7, label=r'$\Delta\chi^2 = 1$ ($1\sigma$)')
ax.axhline(4.0, color='darkorange', ls='--', lw=1.5, alpha=0.7, label=r'$\Delta\chi^2 = 4$ ($2\sigma$)')
ax.axvline(Om_best, color='crimson', ls=':', lw=1.5, alpha=0.7, label=f'Best fit: $\\Omega_m = {Om_best:.3f}$')

ax.set_xlabel(r'$\Omega_m$')
ax.set_ylabel(r'$\Delta\chi^2 = \chi^2(\Omega_m) - \chi^2_{\min}$')
ax.set_title(r'$\chi^2$ Profile for Flat $\Lambda$CDM (Pantheon)')
ax.legend()
ax.set_xlim(0.05, 0.80)
ax.set_ylim(0, 30)

plt.tight_layout()
plt.savefig('chi2_omega_m.png', bbox_inches='tight', dpi=150)
plt.show()

# Find 1-sigma bounds
from scipy.interpolate import interp1d
interp = interp1d(Om_grid, delta_chi2 - 1.0)
from scipy.optimize import brentq
Om_lo = brentq(interp, 0.05, Om_best)
Om_hi = brentq(interp, Om_best, 0.80)
print(f"\n1-sigma constraint: Om = {Om_best:.3f} +{Om_hi-Om_best:.3f} / -{Om_best-Om_lo:.3f}")

---
## Part 4 — Bayesian Inference with MCMC

The $\chi^2$ profile gives us a point estimate and approximate errors. For a full Bayesian analysis, we want the **posterior distribution** of the parameters:

$$P(\boldsymbol{\theta}\,|\,\mathbf{d}) \propto \mathcal{L}(\mathbf{d}\,|\,\boldsymbol{\theta})\;\pi(\boldsymbol{\theta})$$

where:
- $\mathcal{L} = \exp(-\chi^2/2)$ is the **Gaussian likelihood** (up to a normalisation constant)
- $\pi(\boldsymbol{\theta})$ is the **prior** — we use flat (uniform) priors

We sample this posterior using **Markov Chain Monte Carlo (MCMC)** with the `emcee` package, which implements the affine-invariant ensemble sampler of Goodman & Weare (2010). The key advantage: the Bayesian evidence $\mathcal{Z}$ in the denominator of Bayes’ theorem **cancels** in the MCMC acceptance ratio, so we only need the *unnormalised* posterior.

In [ ]:
# ============================================================
# Define log-likelihood, log-prior, and log-posterior for LCDM
# ============================================================

def log_likelihood_LCDM(params):
    """Gaussian log-likelihood: ln L = -chi2 / 2."""
    return -0.5 * chi2_LCDM(params, zcmb, mb, icov)

def log_prior_LCDM(params):
    """Flat priors: Om in (0, 1), MB in (-25, -15)."""
    Om, MB = params
    if 0.0 < Om < 1.0 and -25.0 < MB < -15.0:
        return 0.0
    return -np.inf

def log_posterior_LCDM(params):
    """Log-posterior = log-prior + log-likelihood."""
    lp = log_prior_LCDM(params)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood_LCDM(params)

print(f"ln P at best-fit: {log_posterior_LCDM([Om_best, MB_best]):.1f}")

In [ ]:
# ============================================================
# Run emcee MCMC for flat LCDM (2 parameters)
# ============================================================
ndim_LCDM = 2
nwalkers = 6
nsteps = 1000

# Initialise walkers near the best-fit with small perturbations
p0_LCDM = res_LCDM.x + 1e-3 * np.random.randn(nwalkers, ndim_LCDM)

# Create the sampler and run
sampler_LCDM = emcee.EnsembleSampler(nwalkers, ndim_LCDM, log_posterior_LCDM)

print(f"Running MCMC for flat LCDM ({ndim_LCDM} params, {nwalkers} walkers, {nsteps} steps)...")
sampler_LCDM.run_mcmc(p0_LCDM, nsteps, progress=True)
print("Done.")

In [ ]:
# ============================================================
# Trace plots (chain evolution)
# ============================================================
chain_LCDM = sampler_LCDM.get_chain()  # shape: (nsteps, nwalkers, ndim)
labels_LCDM = [r'$\Omega_m$', r'$M_B$']

fig, axes = plt.subplots(ndim_LCDM, 1, figsize=(10, 5), sharex=True)
for i in range(ndim_LCDM):
    for j in range(nwalkers):
        axes[i].plot(chain_LCDM[:, j, i], alpha=0.3, lw=0.5)
    axes[i].set_ylabel(labels_LCDM[i])
    axes[i].axvline(200, color='crimson', ls='--', lw=1.5, alpha=0.7)

axes[-1].set_xlabel('Step')
axes[0].set_title('MCMC Trace Plots — Flat $\\Lambda$CDM')
# Red line marks approximate burn-in at step 200
axes[0].text(210, axes[0].get_ylim()[1], 'burn-in', color='crimson', fontsize=10)

plt.tight_layout()
plt.show()

print("The chains converge quickly. We discard the first 200 steps as burn-in.")

### Exercise 2: Corner Plot and Parameter Constraints

**Tasks:**
1. Discard the first 200 steps as burn-in and flatten the chains
2. Make a **corner plot** showing the 1D and 2D marginalised posteriors
3. Report the constraints on $\Omega_m$ and $M_B$ as median $\pm$ (84th–50th) / (50th–16th) percentiles

**Hints:**
- `flat_samples = sampler_LCDM.get_chain(discard=200, flat=True)`
- `corner.corner(flat_samples, labels=[...], truths=[...], quantiles=[0.16, 0.5, 0.84])`

In [ ]:
# YOUR CODE HERE: Make corner plot and extract parameter constraints
#
# Tasks:
#   1. Discard the first 200 steps as burn-in
#   2. Flatten the chains into a 2D array
#   3. Make a corner plot using the corner library
#   4. Report Om and MB as median +/- percentiles
#
# Hints:
#   flat_samples = sampler_LCDM.get_chain(discard=200, flat=True)
#   fig = corner.corner(flat_samples, labels=labels_LCDM, truths=res_LCDM.x,
#                        quantiles=[0.16, 0.5, 0.84], show_titles=True)
#   mcmc = np.percentile(flat_samples[:, i], [16, 50, 84])


---
## Part 5 — Dark Energy Equation of State

So far we have assumed $\Lambda$CDM, where dark energy has a fixed equation of state $w = -1$. But what if $w \neq -1$, or if $w$ varies with time?

We now generalise to two dark energy models:

1. **$w$CDM**: constant $w$, with $w = -1$ recovering $\Lambda$CDM
2. **CPL** (Chevallier–Polarski–Linder): $w(a) = w_0 + w_a(1-a)$, the standard parametrisation used by DESI, Euclid, and Rubin/LSST

The dark energy density evolves as:
- **$w$CDM:** $\rho_{\mathrm{DE}} \propto (1+z)^{3(1+w)}$
- **CPL:** $\rho_{\mathrm{DE}} \propto (1+z)^{3(1+w_0+w_a)}\,\exp\!\left(-3w_a\,\frac{z}{1+z}\right)$

In [ ]:
# ============================================================
# wCDM model: 3 parameters (Om, w, MB)
# ============================================================

def chi2_wCDM(params, zcmb, mb, icov):
    """Chi-squared for flat wCDM: params = [Om, w, MB]."""
    Om, w, MB = params
    if Om <= 0 or Om >= 1:
        return 1e10
    try:
        cosmo = Flatw0waCDM(H0=H0_fid, Om0=Om, w0=w, wa=0.0)
        mu_th = cosmo.distmod(zcmb).value
    except Exception:
        return 1e10
    dy = mb - mu_th - MB
    return float(dy @ icov @ dy)

def log_posterior_wCDM(params):
    """Log-posterior for flat wCDM."""
    Om, w, MB = params
    # Flat priors
    if not (0.0 < Om < 1.0 and -3.0 < w < 0.0 and -25.0 < MB < -15.0):
        return -np.inf
    return -0.5 * chi2_wCDM(params, zcmb, mb, icov)

# --- Find starting point ---
print("Finding best-fit wCDM parameters...")
bounds_wCDM = [(0.01, 0.99), (-3.0, -0.01), (-25.0, -15.0)]
res_wCDM = differential_evolution(
    chi2_wCDM, bounds_wCDM, args=(zcmb, mb, icov),
    seed=42, tol=1e-8
)
Om_w, w_w, MB_w = res_wCDM.x
chi2_min_wCDM = res_wCDM.fun

print(f"\nBest-fit flat wCDM:")
print(f"  Om = {Om_w:.4f}")
print(f"  w  = {w_w:.4f}")
print(f"  MB = {MB_w:.4f}")
print(f"  chi2_min = {chi2_min_wCDM:.2f}")

In [ ]:
# ============================================================
# Run MCMC for wCDM (3 parameters)
# ============================================================
ndim_wCDM = 3
nsteps_wCDM = 1000

p0_wCDM = res_wCDM.x + 1e-3 * np.random.randn(nwalkers, ndim_wCDM)
sampler_wCDM = emcee.EnsembleSampler(nwalkers, ndim_wCDM, log_posterior_wCDM)

print(f"Running MCMC for wCDM ({ndim_wCDM} params, {nwalkers} walkers, {nsteps_wCDM} steps)...")
sampler_wCDM.run_mcmc(p0_wCDM, nsteps_wCDM, progress=True)
print("Done.")

In [ ]:
# ============================================================
# Corner plot for wCDM
# ============================================================
flat_wCDM = sampler_wCDM.get_chain(discard=300, flat=True)
labels_wCDM = [r'$\Omega_m$', r'$w$', r'$M_B$']

fig = corner.corner(
    flat_wCDM,
    labels=labels_wCDM,
    truths=[Om_w, w_w, MB_w],
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True,
    title_kwargs={'fontsize': 13},
    color='darkorange'
)
# Mark LCDM point (w = -1)
fig.suptitle('Flat $w$CDM Posterior (Pantheon)', fontsize=15, y=1.02)
plt.savefig('wcdm_corner.png', bbox_inches='tight', dpi=150)
plt.show()

# Report constraints
print("\nwCDM constraints:")
for i, label in enumerate(labels_wCDM):
    mcmc = np.percentile(flat_wCDM[:, i], [16, 50, 84])
    q = np.diff(mcmc)
    print(f"  {label} = {mcmc[1]:.4f} +{q[1]:.4f} / -{q[0]:.4f}")

print(f"\nNote: w is consistent with -1 (LCDM) within the uncertainties.")
print(f"The Om-w degeneracy is visible in the 2D contour.")

### Exercise 3: CPL Parametrisation

The **Chevallier–Polarski–Linder (CPL)** parametrisation allows the equation of state to vary with time:

$$w(a) = w_0 + w_a(1-a) \quad \Longleftrightarrow \quad w(z) = w_0 + w_a\,\frac{z}{1+z}$$

$\Lambda$CDM corresponds to $(w_0, w_a) = (-1, 0)$.

**Tasks:**
1. Define `chi2_CPL` and `log_posterior_CPL` for 4 parameters: $(\Omega_m, w_0, w_a, M_B)$
2. Find the best-fit with `differential_evolution`
3. Run MCMC with `emcee` (12 walkers, 1000 steps)
4. Make a corner plot and report constraints on $w_0$ and $w_a$

**Hints:**
- Use `Flatw0waCDM(H0=70, Om0=Om, w0=w0, wa=wa)` for distance computation
- Priors: $\Omega_m \in (0,1)$, $w_0 \in (-3,0)$, $w_a \in (-3,3)$, $M_B \in (-25,-15)$
- This will take ~3–5 minutes to run

In [ ]:
# YOUR CODE HERE: Run MCMC for the CPL parametrisation (4 parameters)
#
# Tasks:
#   1. Define chi2_CPL(params, zcmb, mb, icov) for params = [Om, w0, wa, MB]
#   2. Define log_posterior_CPL(params) with flat priors:
#      Om in (0, 1), w0 in (-3, 0), wa in (-3, 3), MB in (-25, -15)
#   3. Find starting point with differential_evolution
#   4. Run emcee with 12 walkers and 1000 steps
#   5. Make corner plot and report constraints on w0 and wa
#
# Hints:
#   - Use Flatw0waCDM(H0=70, Om0=Om, w0=w0, wa=wa) for distances
#   - Follow the same pattern as the wCDM code above
#   - This will take ~3-5 minutes to run


In [ ]:
# ============================================================
# Summary comparison: Om posteriors from all three models
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: 1D Om posteriors ---
ax = axes[0]
Om_bins = np.linspace(0.0, 0.8, 60)
ax.hist(flat_LCDM[:, 0], bins=Om_bins, density=True, alpha=0.5,
        color='steelblue', label=r'$\Lambda$CDM', edgecolor='white')
ax.hist(flat_wCDM[:, 0], bins=Om_bins, density=True, alpha=0.5,
        color='darkorange', label=r'$w$CDM', edgecolor='white')
ax.hist(flat_CPL[:, 0], bins=Om_bins, density=True, alpha=0.5,
        color='seagreen', label='CPL', edgecolor='white')
ax.set_xlabel(r'$\Omega_m$')
ax.set_ylabel('Posterior density')
ax.set_title(r'Marginalised $\Omega_m$ Posteriors')
ax.legend()

# --- Right: w0-wa plane from CPL ---
ax = axes[1]
ax.scatter(flat_CPL[:, 1], flat_CPL[:, 2], s=1, alpha=0.05, color='seagreen')

# Mark LCDM point
ax.plot(-1, 0, 'k*', ms=15, zorder=10, label=r'$\Lambda$CDM ($w_0=-1, w_a=0$)')

# Approximate DESI constraints as an ellipse
from matplotlib.patches import Ellipse
for nsig, alpha in [(1, 0.6), (2, 0.3)]:
    ell = Ellipse(xy=(-0.8, -0.7), width=2*0.1*nsig, height=2*0.4*nsig,
                  angle=0, facecolor='none', edgecolor='crimson',
                  ls='--', lw=2, alpha=alpha)
    ax.add_patch(ell)
ax.plot([], [], 'r--', lw=2, label='DESI+CMB+SNe (approx.)')  # legend entry

ax.set_xlabel(r'$w_0$')
ax.set_ylabel(r'$w_a$')
ax.set_title(r'$w_0$–$w_a$ Plane: Pantheon vs DESI')
ax.set_xlim(-2.5, 0.0)
ax.set_ylim(-3.0, 3.0)
ax.legend(fontsize=10)

plt.suptitle('Dark Energy Constraints Comparison', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

---
## Part 6 — Model Selection

Adding more parameters to a model always improves the fit (lower $\chi^2_{\min}$), but is the improvement *statistically significant*? We need to penalise model complexity to avoid overfitting.

Two widely used criteria:

**Akaike Information Criterion (AIC):**
$$\mathrm{AIC} = \chi^2_{\min} + 2p$$

**Bayesian Information Criterion (BIC):**
$$\mathrm{BIC} = \chi^2_{\min} + p\,\ln N$$

where $p$ is the number of free parameters and $N$ is the number of data points. The BIC penalises extra parameters more heavily when $N$ is large (for Pantheon, $\ln 1048 \approx 6.95$).

**Interpretation:** Compare $\Delta$AIC or $\Delta$BIC between models:
- $|\Delta| < 2$: Models are statistically indistinguishable
- $2 < |\Delta| < 6$: Moderate evidence for the model with lower AIC/BIC
- $|\Delta| > 10$: Strong evidence

This embodies the **Bayesian Occam’s razor**: simpler models are preferred unless the data demand additional complexity.

In [ ]:
# ============================================================
# Compute AIC and BIC for all three models
# ============================================================

models = {
    r'Flat $\Lambda$CDM': {'p': 2, 'chi2_min': chi2_min_LCDM},
    r'Flat $w$CDM':       {'p': 3, 'chi2_min': chi2_min_wCDM},
    r'Flat CPL':          {'p': 4, 'chi2_min': chi2_min_CPL},
}

# Reference: LCDM
aic_ref = chi2_min_LCDM + 2 * 2
bic_ref = chi2_min_LCDM + 2 * np.log(N_sne)

print(f"{'Model':<18} {'p':>3} {'chi2_min':>10} {'chi2_red':>10} {'AIC':>10} {'BIC':>10} {'dAIC':>8} {'dBIC':>8}")
print('=' * 82)

for name, info in models.items():
    p = info['p']
    chi2 = info['chi2_min']
    aic = chi2 + 2 * p
    bic = chi2 + p * np.log(N_sne)
    daic = aic - aic_ref
    dbic = bic - bic_ref
    info['AIC'] = aic
    info['BIC'] = bic
    info['dAIC'] = daic
    info['dBIC'] = dbic
    print(f"{name:<18} {p:>3} {chi2:>10.2f} {chi2/(N_sne-p):>10.4f} {aic:>10.2f} {bic:>10.2f} {daic:>+8.2f} {dbic:>+8.2f}")

In [ ]:
# ============================================================
# Visualise model comparison
# ============================================================
model_names = list(models.keys())
daic_vals = [models[m]['dAIC'] for m in model_names]
dbic_vals = [models[m]['dBIC'] for m in model_names]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

x = np.arange(len(model_names))
bar_colors = ['steelblue', 'darkorange', 'seagreen']

ax1.bar(x, daic_vals, color=bar_colors, edgecolor='white', width=0.6)
ax1.axhline(0, color='k', lw=0.8)
ax1.axhline(2, color='gray', ls='--', lw=1, alpha=0.6)
ax1.axhline(-2, color='gray', ls='--', lw=1, alpha=0.6)
ax1.text(2.5, 2.3, 'Inconclusive', fontsize=9, color='gray', ha='right')
ax1.set_ylabel(r'$\Delta$AIC')
ax1.set_title('Akaike Information Criterion')
ax1.set_xticks(x)
ax1.set_xticklabels([r'$\Lambda$CDM', '$w$CDM', 'CPL'], fontsize=11)

ax2.bar(x, dbic_vals, color=bar_colors, edgecolor='white', width=0.6)
ax2.axhline(0, color='k', lw=0.8)
ax2.axhline(2, color='gray', ls='--', lw=1, alpha=0.6)
ax2.axhline(6, color='gray', ls='--', lw=1, alpha=0.6)
ax2.text(2.5, 2.3, 'Moderate', fontsize=9, color='gray', ha='right')
ax2.text(2.5, 6.3, 'Strong', fontsize=9, color='gray', ha='right')
ax2.set_ylabel(r'$\Delta$BIC')
ax2.set_title('Bayesian Information Criterion')
ax2.set_xticks(x)
ax2.set_xticklabels([r'$\Lambda$CDM', '$w$CDM', 'CPL'], fontsize=11)

plt.suptitle(r'Model Selection: $\Lambda$CDM vs Dark Energy Extensions', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('model_selection.png', bbox_inches='tight', dpi=150)
plt.show()

print("\nInterpretation:")
print("  AIC: Small penalty for extra parameters — wCDM and CPL are comparable to LCDM")
print("  BIC: Heavier penalty (ln N ~ 7) — LCDM is clearly preferred")
print("  \nWith Pantheon data alone, there is NO evidence for dark energy evolution.")
print("  LCDM (w = -1) remains the simplest and best-supported model.")

---
## Summary

In this tutorial you have:

1. **Computed cosmological distances** from first principles (comoving, luminosity, angular diameter) and verified them against `astropy`, confirming the Etherington reciprocity relation
2. **Downloaded the Pantheon Type Ia supernova dataset** (1048 SNe Ia) and built the Hubble diagram that reveals cosmic acceleration
3. **Constructed the $\chi^2$ statistic** with the full systematic covariance matrix and found the best-fit matter density for flat $\Lambda$CDM
4. **Performed Bayesian inference with MCMC** using `emcee`, obtaining posterior constraints on $\Omega_m$ and the absolute magnitude $M_B$
5. **Extended to dark energy models** ($w$CDM and CPL) and quantified the dark energy equation of state
6. **Applied model selection criteria** (AIC, BIC) to determine whether the extra complexity of evolving dark energy is justified by current data

### Key Takeaways
- Type Ia supernovae provided the first direct evidence for cosmic acceleration in 1998 (Nobel Prize 2011)
- The full covariance matrix (systematic + statistical) is essential for accurate parameter estimation — ignoring it leads to artificially tight constraints
- Current supernova data alone are consistent with a cosmological constant ($w = -1$), but combined with BAO (DESI), there are hints of evolving dark energy at 2–3$\sigma$
- Model selection criteria (AIC, BIC) penalise unnecessary complexity, implementing the **Bayesian Occam’s razor**

### References
- Lecture 3 slides and notes: *Distances in Cosmology and Dark Energy*
- Appendix A: *Likelihood and Model Selection*
- Scolnic, D.M. et al. 2018, ApJ, **859**, 101 (Pantheon)
- Perlmutter, S. et al. 1999, ApJ, **517**, 565 (discovery of acceleration)
- Riess, A.G. et al. 1998, AJ, **116**, 1009 (discovery of acceleration)
- DESI Collaboration 2024 (BAO Year 1 results)